# Pricing Project

In [11]:
# Problem 1: MNL Model

import numpy as np
import pandas as pd
from scipy.optimize import minimize
from scipy.special import logsumexp

df = pd.read_csv('Data/data.csv')

feature_cols = [
    'prop_starrating', 'prop_review_score', 'prop_brand_bool',
    'prop_location_score', 'prop_accesibility_score',
    'prop_log_historical_price', 'price_usd', 'promotion_flag'
]

# Normalize features
means = df[feature_cols].mean()
stds = df[feature_cols].std()
estimation_df = df.copy()
estimation_df[feature_cols] = (df[feature_cols] - means) / stds

# Group by search session
groups = []
for _, g in estimation_df.groupby('srch_id'):
    groups.append((g[feature_cols].values, g['booking_bool'].values))

def neg_ll(beta):
    b0, b = beta[0], beta[1:]
    nll, grad = 0.0, np.zeros_like(beta)
    for X, y in groups:
        u = b0 + X @ b
        ld = logsumexp(np.append(0.0, u))
        p = np.exp(u - ld)
        d = p - y
        nll += ld - y @ u
        grad[0] += d.sum()
        grad[1:] += X.T @ d
    return nll, grad

result = minimize(neg_ll, x0=np.zeros(9), jac=True, method='L-BFGS-B')
beta_star = result.x

print('Estimated MNL coefficients:')
for name, val in zip(['intercept'] + feature_cols, beta_star):
    print(f'  {name:35s} {val:+f}')

Estimated MNL coefficients:
  intercept                           -1.746680
  prop_starrating                     +0.408132
  prop_review_score                   +0.108754
  prop_brand_bool                     +0.101381
  prop_location_score                 +0.021995
  prop_accesibility_score             +0.043445
  prop_log_historical_price           -0.066872
  price_usd                           -1.331110
  promotion_flag                      +0.159765


In [3]:
# Problem 2: Assortment Optimization under MNL
# Using the MNL estimated in Problem 1 (beta_star) and the same standardization (means/stds).

from pathlib import Path

small_data_paths = {
    "data1": [Path("Data/data1.csv"), Path("data1.csv")],
    "data2": [Path("Data/data2.csv"), Path("data2.csv")],
    "data3": [Path("Data/data3.csv"), Path("data3.csv")],
    "data4": [Path("Data/data4.csv"), Path("data4.csv")],
}


def load_small_dataset(name: str) -> pd.DataFrame:
    candidates = small_data_paths[name]
    path = next((p for p in candidates if p.exists()), None)
    if path is None:
        raise FileNotFoundError(f"Could not find {name}.csv in expected locations: {candidates}")
    return pd.read_csv(path)


def compute_mnl_weights(df_small: pd.DataFrame) -> tuple[np.ndarray, np.ndarray, pd.DataFrame]:
    """Return (v, price, df_with_norm_cols). v_j = exp(u_j), u_j = b0 + x_norm @ b."""
    missing = [c for c in feature_cols if c not in df_small.columns]
    if missing:
        raise KeyError(f"Missing columns in small dataset: {missing}")

    df2 = df_small.copy()
    # Apply the same standardization as in Problem 1
    X_norm = (df2[feature_cols] - means) / stds

    b0, b = float(beta_star[0]), beta_star[1:].astype(float)
    u = b0 + X_norm.to_numpy(dtype=float) @ b
    v = np.exp(u)
    price = df2["price_usd"].to_numpy(dtype=float)
    return v, price, df2


def expected_revenue(v: np.ndarray, price: np.ndarray, mask: np.ndarray) -> float:
    """Expected revenue for offering subset S = {j: mask_j==True} under MNL with outside option."""
    vS = v[mask]
    pS = price[mask]
    denom = 1.0 + vS.sum()
    return float((pS * vS).sum() / denom)


def optimal_assortment_revenue_ordered(v: np.ndarray, price: np.ndarray) -> tuple[np.ndarray, float]:
    """Find optimal assortment using the MNL 'revenue-ordered assortment' property.

    We evaluate assortments of the form S(r) = {j : price_j >= r} for each distinct price level r.
    This avoids cutting through ties.

    Returns (mask, best_rev).
    """
    n = len(price)
    order = np.argsort(-price, kind="mergesort")  # stable sort for reproducibility
    p_sorted = price[order]
    v_sorted = v[order]

    cum_v = np.cumsum(v_sorted)
    cum_pv = np.cumsum(p_sorted * v_sorted)

    # Candidate cut points: end of each price level (include all items with price >= that level)
    cut_idxs = np.flatnonzero(np.r_[p_sorted[1:] != p_sorted[:-1], True])

    best_rev = 0.0
    best_k = -1  # -1 means empty assortment

    for k in cut_idxs:
        denom = 1.0 + cum_v[k]
        rev = float(cum_pv[k] / denom)
        if rev > best_rev:
            best_rev = rev
            best_k = int(k)

    best_mask_sorted = np.zeros(n, dtype=bool)
    if best_k >= 0:
        best_mask_sorted[: best_k + 1] = True

    best_mask = np.zeros(n, dtype=bool)
    best_mask[order] = best_mask_sorted
    return best_mask, best_rev


results = []
for name in ["data1", "data2", "data3", "data4"]:
    df_small = load_small_dataset(name)
    v, price, _ = compute_mnl_weights(df_small)
    mask, rev = optimal_assortment_revenue_ordered(v, price)

    results.append(
        {
            "dataset": name,
            "n_hotels": int(len(df_small)),
            "n_offered": int(mask.sum()),
            "expected_revenue": rev,
        }
    )

    offered_idx = np.flatnonzero(mask)
    offered_prices = price[mask]

    print(f"\n{name}: optimal offered subset size = {mask.sum()} out of {len(df_small)}")
    print(f"{name}: expected revenue = {rev:.6f}")
    print(f"{name}: offered row indices (0-based) = {offered_idx.tolist()}")
    print(f"{name}: offered prices = {np.round(np.sort(offered_prices), 2).tolist()}")

summary_df = pd.DataFrame(results).sort_values("dataset")
print("\nSummary")
display(summary_df)


data1: optimal offered subset size = 18 out of 27
data1: expected revenue = 107.352012
data1: offered row indices (0-based) = [0, 1, 2, 3, 4, 5, 6, 12, 15, 17, 18, 19, 20, 21, 22, 23, 24, 26]
data1: offered prices = [108.0, 112.0, 118.0, 120.0, 121.0, 121.0, 125.0, 131.0, 131.0, 140.0, 144.0, 145.0, 147.0, 147.0, 150.0, 154.0, 156.0, 158.0]

data2: optimal offered subset size = 10 out of 29
data2: expected revenue = 131.110898
data2: offered row indices (0-based) = [0, 1, 6, 7, 8, 9, 10, 21, 23, 25]
data2: offered prices = [133.0, 134.0, 137.0, 143.0, 146.0, 147.0, 161.0, 206.0, 212.0, 233.0]

data3: optimal offered subset size = 18 out of 26
data3: expected revenue = 121.053221
data3: offered row indices (0-based) = [0, 1, 2, 3, 4, 5, 7, 8, 10, 11, 13, 14, 15, 16, 18, 19, 23, 24]
data3: offered prices = [123.0, 128.0, 129.0, 138.0, 138.0, 140.0, 144.0, 150.0, 153.0, 171.0, 180.0, 181.0, 190.0, 191.0, 195.0, 211.0, 281.0, 603.0]

data4: optimal offered subset size = 11 out of 27
data4

,dataset,n_hotels,n_offered,expected_revenue
0,data1,27,18,107.352012
1,data2,29,10,131.110898
2,data3,26,18,121.053221
3,data4,27,11,97.407947


In [ ]:
# Problem 3: Pricing under MNL
# We display ALL hotels and choose prices to maximize expected revenue.

price_col = "price_usd"
price_idx = feature_cols.index(price_col)

beta0 = float(beta_star[0])
beta = beta_star[1:].astype(float)
beta_price = float(beta[price_idx])
mean_price = float(means[price_col])
std_price = float(stds[price_col])


def optimize_prices_mnl(df_small: pd.DataFrame, *, lower: float, upper: float) -> dict:
    missing = [c for c in feature_cols if c not in df_small.columns]
    if missing:
        raise KeyError(f"Missing columns in small dataset: {missing}")

    X = df_small[feature_cols].to_numpy(dtype=float)

    # Standardize non-price features using Problem 1 means/stds
    z = (X - means[feature_cols].to_numpy(dtype=float)) / stds[feature_cols].to_numpy(dtype=float)

    # Split out price term so we can vary p
    z_no_price = z.copy()
    z_no_price[:, price_idx] = 0.0

    # u(p) = base + beta_price * ((p - mean_price)/std_price)
    base = beta0 + z_no_price @ beta

    def neg_rev_and_grad(p: np.ndarray) -> tuple[float, np.ndarray]:
        # enforce numeric dtype
        p = p.astype(float)
        u = base + beta_price * ((p - mean_price) / std_price)
        v = np.exp(u)
        D = 1.0 + v.sum()
        N = float((p * v).sum())
        rev = N / D

        # gradient
        c = beta_price / std_price  # dv_j/dp_j = c * v_j
        dv = c * v
        dN = v + p * dv
        dD = dv
        grad = (dN * D - N * dD) / (D * D)

        return -rev, -grad

    def neg_rev(p: np.ndarray) -> float:
        return neg_rev_and_grad(p)[0]

    def neg_rev_grad(p: np.ndarray) -> np.ndarray:
        return neg_rev_and_grad(p)[1]

    p0 = df_small[price_col].to_numpy(dtype=float)
    bounds = [(lower, upper) for _ in range(len(p0))]

    res = minimize(
        neg_rev,
        p0,
        jac=neg_rev_grad,
        bounds=bounds,
        method="L-BFGS-B",
    )

    if not res.success:
        raise RuntimeError(f"Pricing optimization failed: {res.message}")

    p_star = res.x.astype(float)

    # baseline revenue at current prices
    base_u0 = base + beta_price * ((p0 - mean_price) / std_price)
    v0 = np.exp(base_u0)
    rev0 = float((p0 * v0).sum() / (1.0 + v0.sum()))

    # optimized revenue
    base_u_star = base + beta_price * ((p_star - mean_price) / std_price)
    v_star = np.exp(base_u_star)
    rev_star = float((p_star * v_star).sum() / (1.0 + v_star.sum()))

    return {
        "p0": p0,
        "p_star": p_star,
        "rev0": rev0,
        "rev_star": rev_star,
        "optimizer_message": res.message,
        "n_iters": int(res.nit),
    }


pricing_results = []
for name in ["data1", "data2", "data3", "data4"]:
    df_small = load_small_dataset(name)

    p0 = df_small[price_col].to_numpy(dtype=float)
    lo = 1.0
    hi = max(1000.0, 3.0 * float(np.max(p0)))

    out = optimize_prices_mnl(df_small, lower=lo, upper=hi)

    pricing_results.append(
        {
            "dataset": name,
            "n_hotels": int(len(df_small)),
            "rev_current": out["rev0"],
            "rev_opt": out["rev_star"],
            "rev_lift": out["rev_star"] - out["rev0"],
            "price_min_current": float(np.min(out["p0"])),
            "price_max_current": float(np.max(out["p0"])),
            "price_min_opt": float(np.min(out["p_star"])),
            "price_max_opt": float(np.max(out["p_star"])),
            "iters": out["n_iters"],
        }
    )

    print(f"\n{name}: expected revenue (current prices) = {out['rev0']:.6f}")
    print(f"{name}: expected revenue (optimal prices) = {out['rev_star']:.6f}")
    print(f"{name}: optimizer status = {out['optimizer_message']}")
    print(f"{name}: optimal prices (rounded) = {np.round(out['p_star'], 2).tolist()}")

pricing_summary = pd.DataFrame(pricing_results).sort_values("dataset")
print("\nPricing summary")
display(pricing_summary)

In [10]:
# Problem 7: AI agents as customers

df_agent = pd.read_csv('Data/data_empty_bool.csv')

feature_cols = [
    'prop_starrating', 'prop_review_score', 'prop_brand_bool',
    'prop_location_score', 'prop_accesibility_score',
    'prop_log_historical_price', 'price_usd', 'promotion_flag'
]

# Normalize features
means = df_agent[feature_cols].mean()
stds = df_agent[feature_cols].std()
estimation_df = df_agent.copy()
estimation_df[feature_cols] = (df_agent[feature_cols] - means) / stds

# Group by search session
groups = []
for _, g in estimation_df.groupby('srch_id'):
    groups.append((g[feature_cols].values, g['booking_bool'].values))

def neg_ll(beta):
    b0, b = beta[0], beta[1:]
    nll, grad = 0.0, np.zeros_like(beta)
    for X, y in groups:
        u = b0 + X @ b
        ld = logsumexp(np.append(0.0, u))
        p = np.exp(u - ld)
        d = p - y
        nll += ld - y @ u
        grad[0] += d.sum()
        grad[1:] += X.T @ d
    return nll, grad

result = minimize(neg_ll, x0=np.zeros(9), jac=True, method='L-BFGS-B')
beta_star = result.x

print('Reestimated MNL coefficients on agent created dataset:')
for name, val in zip(['intercept'] + feature_cols, beta_star):
    print(f'  {name:35s} {val:+f}')


Reestimated MNL coefficients on agent created dataset:
  intercept                           -5.778389
  prop_starrating                     +0.315486
  prop_review_score                   +0.717503
  prop_brand_bool                     +0.157374
  prop_location_score                 +0.074431
  prop_accesibility_score             +0.102504
  prop_log_historical_price           +0.000662
  price_usd                           -0.981444
  promotion_flag                      +0.529794


**Difference between human and AI agent as customers**
Both models agree on sign for 7 of 8 features - star rating, review score, brand, location, accessibility, and price all move utility in the same direction, confirming the agent captures the broad structure of human preferences. The agent's intercept is far more negative (-5.78 vs -1.75), meaning it chose the no-purchase option much more often than real customers. The agent overweights review_score (~6.6x larger) and promotion_flag (~3.3x larger) relative to humans, suggesting it reasons more "rationally" about quality signals. Price sensitivity is actually lower for the agent (-0.98 vs -1.33), indicating real humans are more price-driven than the AI simulates. The one sign flip is log_historical_price (human: -0.067, agent: ~0.0) - humans slightly penalize expensive hotels while the agent is indifferent to price history.    

**Model and Prompts used**
- Opus 4.6 extended model
- Initial prompt: "You are simulating a customer searching for a hotel on Expedia. You will be given the customer's travel context and a list of available hotels with their attributes. Choose either one hotel to book, or choose "no purchase”. Respond in column O (booking_bool) with only a single integer: 1 if you choose a hotel for a customer, or 0 for no purchase. Make sure you assign a maximum of one hotel to each customer, but it can also be none if the option is not appealing."
- Prompt for reasoning explanation below: "Please give me 2 paragraphs explaining the reasoning behind whether a customer booked a room or not."

**Reasoning of the model**
The simulated customer evaluates each hotel in their search results by weighing quality against price. Quality is a composite of star rating (weighted most heavily), review score, location score, brand recognition, and accessibility. This quality signal is then balanced against how the hotel's price compares both to other options in the same search and to its own historical pricing — a hotel priced well below its historical average signals a good deal, while one priced above the average for the search set gets penalized. Promotions provide an additional nudge, reflecting how real customers are drawn to visible discounts. The customer then identifies the single best-value option from the set, but filters out any hotel that falls below a baseline threshold (at least 2 stars and a review score of 2), since most travelers won't book something that looks outright poor regardless of price.
Even when a strong option exists, the customer doesn't always book — most hotel searches are exploratory, with travelers comparing across sessions, dates, or platforms before committing. To capture this, the model applies a probabilistic conversion rate in the range of roughly 4–8%, nudged upward when the best available hotel is especially compelling relative to the rest of the set. This yields an overall 8.4% booking rate, which aligns with realistic search-to-book conversion on travel platforms. The net effect is that booked hotels tend to be the ones offering the best combination of high reviews, solid star ratings, competitive pricing, and active promotions — exactly the kind of standout listing that would convert a browsing customer into a paying one.


Same prompt as above, but with the additional context and 80/20 train/test split of data. Added to the beginning of the prompt "Use train_80.csv as context to predict the empty booking_bool column in test_20.csv."

In [12]:
# AUC-ROC comparison: MNL (Problem 1) vs AI Agent predictions
from sklearn.metrics import roc_auc_score

# Re-estimate Problem 1 MNL (cell-4 overwrote beta_star/means/stds)
df_full = pd.read_csv('Data/data.csv')

feature_cols = [
    'prop_starrating', 'prop_review_score', 'prop_brand_bool',
    'prop_location_score', 'prop_accesibility_score',
    'prop_log_historical_price', 'price_usd', 'promotion_flag'
]

means_p1 = df_full[feature_cols].mean()
stds_p1 = df_full[feature_cols].std()
df_full[feature_cols] = (df_full[feature_cols] - means_p1) / stds_p1

groups_p1 = []
for _, g in df_full.groupby('srch_id'):
    groups_p1.append((g[feature_cols].values, g['booking_bool'].values))

def neg_ll_p1(beta):
    b0, b = beta[0], beta[1:]
    nll, grad = 0.0, np.zeros_like(beta)
    for X, y in groups_p1:
        u = b0 + X @ b
        ld = logsumexp(np.append(0.0, u))
        p = np.exp(u - ld)
        d = p - y
        nll += ld - y @ u
        grad[0] += d.sum()
        grad[1:] += X.T @ d
    return nll, grad

res_p1 = minimize(neg_ll_p1, x0=np.zeros(9), jac=True, method='L-BFGS-B')
beta_p1 = res_p1.x

# Load test data
truth = pd.read_csv('Data/test_ground_truth.csv')
pred = pd.read_csv('Data/test_20_predicted.csv')

y_true = truth['booking_bool'].values
y_agent = pred['booking_bool'].values

# MNL predicted probabilities on test set
test_norm = (pred[feature_cols] - means_p1) / stds_p1

mnl_probs = np.zeros(len(pred))
for srch_id, g in pred.groupby('srch_id'):
    idx = g.index.values
    X = test_norm.loc[idx, feature_cols].values
    u = beta_p1[0] + X @ beta_p1[1:]
    ld = logsumexp(np.append(0.0, u))
    mnl_probs[idx] = np.exp(u - ld)

# AUC-ROC
auc_mnl = roc_auc_score(y_true, mnl_probs)
auc_agent = roc_auc_score(y_true, y_agent)

print(f'AUC-ROC (MNL Problem 1):  {auc_mnl:.4f}')
print(f'AUC-ROC (AI Agent):       {auc_agent:.4f}')

AUC-ROC (MNL Problem 1):  0.7420
AUC-ROC (AI Agent):       0.5617


**Metric: AUC-ROC**

We use AUC-ROC (Area Under the Receiver Operating Characteristic curve) to compare both models. AUC-ROC measures how well a model ranks actual bookings above non-bookings, on a scale from 0.5 (random) to 1.0 (perfect). We chose this metric because the data is heavily imbalanced - only ~7% of rows are bookings - making raw accuracy misleading (always predicting "no booking" gives 93% accuracy but zero predictive value). AUC-ROC is threshold-independent, meaning it evaluates the model's ranking ability rather than requiring us to pick an arbitrary cutoff for what counts as a "predicted booking."

**Setup and Results**

We split the real data 80/20. The AI agent (Claude Opus 4.6) received the training set as context and predicted booking decisions on the held-out test set. The MNL from Problem 1 produced choice probabilities P(j|S) on the same test queries.

| Model | AUC-ROC |
|---|---|
| MNL (Problem 1) | 0.7420 |
| AI Agent (Opus 4.6) | 0.5617 |

**Discussion**

The MNL substantially outperforms the agent (0.74 vs 0.56, barely above random). The MNL learns exact statistical tradeoffs from observed choices through maximum likelihood, while the agent relies on general reasoning about hotel quality. As shown in the coefficient comparison, the agent captures directional preferences (7 of 8 coefficient signs match), but cannot calibrate the relative magnitudes needed for accurate prediction.

These results suggest that AI agents can approximate the *qualitative structure* of human hotel choice behavior - understanding that higher stars, better reviews, and lower prices make a hotel more attractive - but fail at *quantitative prediction*. Real human choices involve idiosyncratic tradeoffs, reference-dependent price sensitivity, and contextual factors (e.g., how a hotel's price compares to others in the same search) that general-purpose reasoning cannot replicate. A simple 9-parameter MNL model fit to revealed preference data significantly outperforms a state-of-the-art LLM at predicting which specific hotel a customer will book. This implies that for structured choice settings like hotel search, statistical models trained on observed behavior remain the right tool for prediction, while AI agents are better suited for generating plausible synthetic choice data or exploring "what if" scenarios where real data is unavailable.